# 07 — Posición relativa de ingreso por adulto equivalente (medición B, sin línea de pobreza)

Medición B del objetivo: posición relativa de cada hogar frente a la
mediana de `y_ae = ITF/unidades_ae` (nacional y de aglomerado-trimestre),
sin canasta ni línea de pobreza oficial -- a diferencia de la medición A
(`05_distancia_linea_pobreza.ipynb`, 2016T2-2025T4), cubre **toda la serie
2003T3-2025T4**. El fundamento matemático (por qué el componente propio de
la variación coincide con el de A en el solape) y sus límites quedan en
`.claude/decisiones.md`.

**Lee** `data/02_nucleo/{hogar,individual}`. **Escribe**
`data/02c_posicion_relativa/hogar/` (Parquet) y
`data/meta/posicion_relativa_reporte_capa1.csv`.

**No toca** `data/02b_distancia_lp` ni ningún artefacto de la medición A.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
from armonizacion import trimestres_publicados, es_historico
import posicion_relativa

TRIMESTRES: list[tuple[int, int]] | None = None
REHACER: bool = False

NUCLEO = REPO / "data" / "02_nucleo"
META = REPO / "data" / "meta"
POSICION_RELATIVA = REPO / "data" / "02c_posicion_relativa" / "hogar"

TODOS_LOS_TRIMESTRES = trimestres_publicados()
trimestres_a_procesar = TRIMESTRES if TRIMESTRES is not None else TODOS_LOS_TRIMESTRES
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
COMMIT_GIT = pm.obtener_commit_git(REPO)
t0_proceso = time.time()
print(f"{len(trimestres_a_procesar)} trimestres de origen a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))

## `data/02c_posicion_relativa/hogar/` (Capa 1 de B)

Por cada trimestre, `posicion_relativa.calcular_posicion_relativa` arma
`y_ae`, las medianas (nacional y por aglomerado) y la posición relativa.
`REHACER=False` reusa una partición existente salvo que `02_nucleo/hogar`
sea más nueva (mismo criterio de staleness que la notebook `01b`).

In [ ]:
COLUMNAS_HOGAR_NUCLEO = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "ITF", "PONDIH",
                          "ingreso_no_declarado"]
COLUMNAS_INDIVIDUAL_NUCLEO = ["CODUSU", "NRO_HOGAR", "CH04", "CH06", "PONDERA"]


def construir_filas_posicion(anio: int, trimestre: int) -> pd.DataFrame | None:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR_NUCLEO, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL_NUCLEO, NUCLEO)
    if hogar is None or individual is None:
        return None
    return posicion_relativa.calcular_posicion_relativa(hogar, individual, es_historico(anio, trimestre))

In [ ]:
def _ruta_posicion(anio: int, trimestre: int) -> Path:
    return POSICION_RELATIVA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"


POSICION_RELATIVA.mkdir(parents=True, exist_ok=True)
filas_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
t0_loop = time.time()
n_reusadas = 0
for anio, trimestre in trimestres_a_procesar:
    origen = pm.ruta_particion(NUCLEO, "hogar", anio, trimestre)
    destino = _ruta_posicion(anio, trimestre)
    if not REHACER and not pm.particion_desactualizada(origen, destino):
        filas_por_trimestre[(anio, trimestre)] = pd.read_parquet(destino)
        n_reusadas += 1
        continue
    filas = construir_filas_posicion(anio, trimestre)
    if filas is None:
        continue
    filas_por_trimestre[(anio, trimestre)] = filas
    destino.parent.mkdir(parents=True, exist_ok=True)
    filas.reset_index(drop=True).to_parquet(destino, index=False)

duracion_loop = time.time() - t0_loop
n_filas = sum(len(df) for df in filas_por_trimestre.values())
print(f"{len(filas_por_trimestre)} trimestres ({n_reusadas} reusados sin recalcular), "
      f"{n_filas:,} filas, {duracion_loop:.1f}s")

pm.registrar_corrida(
    META / "corridas.csv", "07_posicion_relativa_sin_lp",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None},
    len(filas_por_trimestre), n_filas, duracion_loop, COMMIT_GIT,
)

## Reporte de Capa 1, por era

Cobertura de `hot_deck_disponible`, hogares con `y_ae` nulo o `<=0`
(excluidos de cualquier variación de B), y distribución de la posición
relativa de aglomerado -- separado por `era` porque no son comparables
entre sí (ver `.claude/decisiones.md`).

In [ ]:
def _resumen_era(df: pd.DataFrame) -> dict:
    y_ae = pd.to_numeric(df["y_ae"], errors="coerce")
    pos = pd.to_numeric(df["posicion_relativa_aglomerado"], errors="coerce").dropna()
    percentiles = pos.quantile([0.1, 0.25, 0.5, 0.75, 0.9]) if len(pos) else pd.Series(dtype=float)
    return {
        "n_hogares": len(df),
        "pct_hot_deck_disponible": 100 * df["hot_deck_disponible"].mean(),
        "n_y_ae_nulo": int(y_ae.isna().sum()),
        "n_y_ae_no_positivo": int(((y_ae <= 0) & y_ae.notna()).sum()),
        **{f"posicion_aglomerado_p{int(q * 100)}": percentiles.get(q, float("nan")) for q in percentiles.index},
    }


todo_capa1 = pd.concat(filas_por_trimestre.values(), ignore_index=True)
reporte_capa1 = pd.DataFrame({era: _resumen_era(grupo) for era, grupo in todo_capa1.groupby("era")}).T
reporte_capa1.index.name = "era"
reporte_capa1.to_csv(META / "posicion_relativa_reporte_capa1.csv")
print(f"{len(todo_capa1):,} filas totales -- data/meta/posicion_relativa_reporte_capa1.csv")
reporte_capa1